# 04 — LazyPredict ile Model Taraması ve Final Regresyon Modeli

**Hedef:** Mordred 2D feature matrisiyle bir train/test ayrımı oluşturmak, LazyPredict kullanarak çok sayıda regresyon algoritmasını aynı veri üzerinde karşılaştırmak ve SHAP analizine uygun final bir ağaç modelini eğitip kaydetmek.

### Bu notebookta öğreneceklerimiz
- Feature/target ayrımı
- Train/test split ve veri sızıntısı kavramı
- LazyPredict ile hızlı model karşılaştırması
- R², RMSE ve MAE yorumlama
- 5-fold cross-validation
- Final model ve tahmin çıktılarının kaydedilmesi

### Ders planındaki yeri
**Ders 3:** Makine öğrenmesi modelleme ve performans değerlendirme.

## Workshop dosya yapısı

Bu seri çalıştırıldığında Google Drive altında aşağıdaki yapı oluşur:

```text
MyDrive/
└── CHEMBL206_workshop/
    ├── 01_data/
    ├── 02_cleaned/
    ├── 03_features/
    ├── 04_models/
    └── 05_reports/
```

> **Çalıştırma sırası:** Notebook 01 → 02 → 03 → 04 → 05  
> Her kod hücresinin sonunda bir **CHECKPOINT** mesajı vardır. Bir hücre hata verirse sonraki hücreye geçmeyiniz.

## Bilimsel not: LazyPredict ne yapar?

LazyPredict çok sayıda algoritmayı varsayılan ayarlarla hızlıca karşılaştırır. Bu sonuç:

- **nihai hiperparametre optimizasyonu değildir,**
- algoritma ailesi seçmek için bir **başlangıç taramasıdır,**
- test setine tekrar tekrar bakılarak manuel optimizasyon yapılmamalıdır.

Notebook, LazyPredict tablosundan SHAP ile açıklanabilen ağaç modelleri arasındaki en iyi adayı seçer. Desteklenen aday bulunamazsa güvenli varsayılan olarak ExtraTreesRegressor kullanılır.

In [ ]:
# Google Drive bağlantısını kuruyoruz.
from google.colab import drive

# Drive'ı standart Colab yoluna bağlıyoruz.
drive.mount("/content/drive")

print("✅ CHECKPOINT 04.1: Google Drive bağlantısı kuruldu.")

In [ ]:
# LazyPredict'in güncel workshop sürümünü ve modelleme paketlerini kuruyoruz.
!pip -q install lazypredict==0.3.0 scikit-learn joblib matplotlib

print("✅ CHECKPOINT 04.2: LazyPredict ve modelleme paketleri kuruldu.")

In [ ]:
# Dosya yolları için pathlib kullanıyoruz.
from pathlib import Path

# Tablo işlemleri için pandas kullanıyoruz.
import pandas as pd

# Sayısal matris işlemleri için numpy kullanıyoruz.
import numpy as np

# Model metadata kaydı için json kullanıyoruz.
import json

# Eğitilmiş modeli kaydetmek için joblib kullanıyoruz.
import joblib

# Grafik çizimi için matplotlib kullanıyoruz.
import matplotlib.pyplot as plt

# Train/test ayrımı ve çapraz doğrulama araçlarını içe aktarıyoruz.
from sklearn.model_selection import train_test_split, KFold, cross_validate

# Performans metriklerini içe aktarıyoruz.
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Final model adaylarını içe aktarıyoruz.
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)

# LazyPredict regresyon sınıfını içe aktarıyoruz.
from lazypredict.Supervised import LazyRegressor

# Notebook içinde tablo göstermek için display kullanıyoruz.
from IPython.display import display

# Proje klasörlerini tanımlıyoruz.
PROJECT_ROOT = Path("/content/drive/MyDrive/CHEMBL206_workshop")
FEATURE_DIR = PROJECT_ROOT / "03_features"
MODEL_DIR = PROJECT_ROOT / "04_models"
REPORT_DIR = PROJECT_ROOT / "05_reports"

# Çıktı klasörlerini oluşturuyoruz.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Notebook 03 çıktısını giriş dosyası olarak tanımlıyoruz.
INPUT_CSV = FEATURE_DIR / "CHEMBL206_Mordred2D_model_ready.csv"

# Tekrar üretilebilirlik ayarlarını tanımlıyoruz.
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# Giriş dosyasını doğruluyoruz.
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        "Notebook 03 çıktısı bulunamadı. Önce Notebook 03'ü çalıştırınız:\n"
        f"{INPUT_CSV}"
    )

print("Giriş CSV:", INPUT_CSV)
print("Model klasörü:", MODEL_DIR)
print("RANDOM_STATE:", RANDOM_STATE)
print("TEST_SIZE:", TEST_SIZE)
print("✅ CHECKPOINT 04.3: Modelleme yolları ve ayarları hazır.")

In [ ]:
# Model-ready veri setini okuyoruz.
df = pd.read_csv(INPUT_CSV, low_memory=False)

# Target sütununu tanımlıyoruz.
TARGET_COLUMN = "pIC50"

# Model girdisi olmaması gereken metadata sütunlarını tanımlıyoruz.
METADATA_CANDIDATES = [
    "molecule_id",
    "parent_smiles",
    "smiles_raw_example",
    "pIC50",
    "n_records",
    "pIC50_std",
    "pIC50_range",
    "activity_conflict_flag",
]

# Veri setinde bulunan metadata sütunlarını belirliyoruz.
metadata_columns = [
    column
    for column in METADATA_CANDIDATES
    if column in df.columns
]

# Target varlığını kontrol ediyoruz.
if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target sütunu bulunamadı: {TARGET_COLUMN}")

# Feature sütunlarını metadata dışındaki sütunlar olarak belirliyoruz.
feature_columns = [
    column
    for column in df.columns
    if column not in metadata_columns
]

# Feature matrisini float32 biçimine çeviriyoruz.
X = df[feature_columns].astype(np.float32)

# Target dizisini sayısal biçime çeviriyoruz.
y = pd.to_numeric(df[TARGET_COLUMN], errors="coerce").astype(float)

# Eksik target varsa durduruyoruz.
assert y.notna().all(), "Target sütununda eksik değer var."

# Feature matrisinde eksik/sonsuz değer olup olmadığını kontrol ediyoruz.
assert np.isfinite(X.to_numpy()).all(), "Feature matrisinde NaN veya sonsuz değer var."

print("Molekül sayısı:", len(df))
print("Feature sayısı:", len(feature_columns))
print("Target aralığı:", (float(y.min()), float(y.max())))

print("✅ CHECKPOINT 04.4: Feature matrisi ve target dizisi hazır.")

In [ ]:
# Satır indekslerini train/test ayrımı için kullanıyoruz.
all_indices = np.arange(len(df))

# Molekülleri %80 eğitim ve %20 test olacak biçimde ayırıyoruz.
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

# Feature ve target alt kümelerini oluşturuyoruz.
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

# Eğitim ve test metadata tablolarını oluşturuyoruz.
train_meta = df.iloc[train_indices][metadata_columns].copy()
test_meta = df.iloc[test_indices][metadata_columns].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Train ve test indekslerinin çakışmadığını kontrol ediyoruz.
assert set(train_indices).isdisjoint(set(test_indices))

# Tüm satırların tam olarak bir alt kümeye girdiğini kontrol ediyoruz.
assert len(train_indices) + len(test_indices) == len(df)

print("✅ CHECKPOINT 04.5: Train/test ayrımı başarıyla tamamlandı.")

In [ ]:
# LazyPredict regresyon tarayıcısını oluşturuyoruz.
# ignore_warnings=True: Bir model başarısız olursa tüm taramanın durmasını engeller.
lazy_regressor = LazyRegressor(
    verbose=0,
    ignore_warnings=True,
    custom_metric=None,
    predictions=True,
)

# LazyPredict ile varsayılan regresyon algoritmalarını çalıştırıyoruz.
# İlk çıktı model metriklerini, ikinci çıktı test tahminlerini içerir.
lazy_models, lazy_predictions = lazy_regressor.fit(
    X_train,
    X_test,
    y_train,
    y_test,
)

# Sonuç tablosunu ekranda gösteriyoruz.
display(lazy_models.head(20))

# Sonuçları CSV olarak kaydediyoruz.
LAZY_RESULTS_OUTPUT = MODEL_DIR / "CHEMBL206_LazyPredict_regression_results.csv"
LAZY_PREDICTIONS_OUTPUT = MODEL_DIR / "CHEMBL206_LazyPredict_test_predictions.csv"

# Model adını normal bir sütuna dönüştürerek kaydediyoruz.
lazy_models.reset_index().rename(
    columns={"index": "Model"}
).to_csv(LAZY_RESULTS_OUTPUT, index=False)

# Tahmin tablosunu kaydediyoruz.
lazy_predictions.to_csv(LAZY_PREDICTIONS_OUTPUT, index=True)

print("LazyPredict sonuçları:", LAZY_RESULTS_OUTPUT)
print("LazyPredict tahminleri:", LAZY_PREDICTIONS_OUTPUT)

assert len(lazy_models) > 0, "LazyPredict hiçbir model sonucu üretmedi."

print("✅ CHECKPOINT 04.6: LazyPredict model taraması tamamlandı.")

In [ ]:
# LazyPredict tablosundaki R² ve RMSE sütunlarını isim farklılıklarına dayanıklı biçimde buluyoruz.
r2_candidates = ["R-Squared", "R²", "R2"]
rmse_candidates = ["RMSE", "Root Mean Squared Error"]

# Var olan ilk R² sütununu seçiyoruz.
r2_column = next(
    (column for column in r2_candidates if column in lazy_models.columns),
    None,
)

# Var olan ilk RMSE sütununu seçiyoruz.
rmse_column = next(
    (column for column in rmse_candidates if column in lazy_models.columns),
    None,
)

# İlk 15 modeli çizim için seçiyoruz.
plot_df = lazy_models.head(15).copy()

# R² sütunu bulunduysa yatay bar grafiği çiziyoruz.
if r2_column is not None:
    plt.figure(figsize=(9, 6))
    plot_df.sort_values(r2_column)[r2_column].plot(kind="barh")
    plt.xlabel("Test R²")
    plt.ylabel("Model")
    plt.title("LazyPredict — En iyi 15 modelin Test R² değerleri")
    plt.tight_layout()

    # Grafiği Drive'a kaydediyoruz.
    r2_plot_path = REPORT_DIR / "CHEMBL206_LazyPredict_top15_R2.png"
    plt.savefig(r2_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

# RMSE sütunu bulunduysa yatay bar grafiği çiziyoruz.
if rmse_column is not None:
    plt.figure(figsize=(9, 6))
    plot_df.sort_values(rmse_column, ascending=False)[rmse_column].plot(kind="barh")
    plt.xlabel("Test RMSE")
    plt.ylabel("Model")
    plt.title("LazyPredict — En iyi 15 modelin Test RMSE değerleri")
    plt.tight_layout()

    # Grafiği Drive'a kaydediyoruz.
    rmse_plot_path = REPORT_DIR / "CHEMBL206_LazyPredict_top15_RMSE.png"
    plt.savefig(rmse_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

print("Bulunan R² sütunu:", r2_column)
print("Bulunan RMSE sütunu:", rmse_column)
print("✅ CHECKPOINT 04.7: LazyPredict performans grafikleri oluşturuldu.")

In [ ]:
# SHAP ile güvenilir biçimde açıklanabilen üç ağaç modeli için kurucu fonksiyonları tanımlıyoruz.
tree_model_builders = {
    "ExtraTreesRegressor": lambda: ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "RandomForestRegressor": lambda: RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "GradientBoostingRegressor": lambda: GradientBoostingRegressor(
        random_state=RANDOM_STATE,
    ),
}

# LazyPredict sonuçlarının model adlarını metne çeviriyoruz.
lazy_model_names = [str(name) for name in lazy_models.index]

# Sonuç sırasındaki ilk desteklenen ağaç modelini seçiyoruz.
selected_model_name = next(
    (
        model_name
        for model_name in lazy_model_names
        if model_name in tree_model_builders
    ),
    "ExtraTreesRegressor",
)

# Seçilen model nesnesini oluşturuyoruz.
final_model = tree_model_builders[selected_model_name]()

# Final modeli yalnızca eğitim verisinde fit ediyoruz.
final_model.fit(X_train, y_train)

# Eğitim ve test tahminlerini üretiyoruz.
train_predictions = final_model.predict(X_train)
test_predictions = final_model.predict(X_test)

print("Seçilen final model:", selected_model_name)
print("✅ CHECKPOINT 04.8: Final ağaç modeli eğitim verisi üzerinde fit edildi.")

In [ ]:
# Regresyon metriklerini tek sözlükte hesaplayan yardımcı fonksiyonu tanımlıyoruz.
def regression_metrics(y_true, y_pred):
    # R² değerini hesaplıyoruz.
    r2 = r2_score(y_true, y_pred)

    # RMSE için önce MSE, sonra karekök hesaplıyoruz.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAE değerini hesaplıyoruz.
    mae = mean_absolute_error(y_true, y_pred)

    # Sonuçları sözlük olarak döndürüyoruz.
    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
    }


# Eğitim metriklerini hesaplıyoruz.
train_metrics = regression_metrics(y_train, train_predictions)

# Test metriklerini hesaplıyoruz.
test_metrics = regression_metrics(y_test, test_predictions)

# Sonuçları karşılaştırmalı tabloya dönüştürüyoruz.
metrics_table = pd.DataFrame(
    [train_metrics, test_metrics],
    index=["Train", "Test"],
)

# Tabloyu ekranda gösteriyoruz.
display(metrics_table)

print("✅ CHECKPOINT 04.9: Train ve test regresyon metrikleri hesaplandı.")

In [ ]:
# Tekrar üretilebilir 5-fold çapraz doğrulama bölücüsünü tanımlıyoruz.
cv = KFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Çapraz doğrulamada hesaplanacak metrikleri tanımlıyoruz.
scoring = {
    "R2": "r2",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
}

# Final model için yalnızca eğitim setinde çapraz doğrulama yapıyoruz.
cv_results = cross_validate(
    tree_model_builders[selected_model_name](),
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

# Negatif sklearn hata skorlarını pozitif değerlere çeviriyoruz.
cv_r2 = cv_results["test_R2"]
cv_rmse = -cv_results["test_RMSE"]
cv_mae = -cv_results["test_MAE"]

# Ortalama ± standart sapma özetini hazırlıyoruz.
cv_summary = pd.DataFrame(
    {
        "metric": ["R2", "RMSE", "MAE"],
        "mean": [
            cv_r2.mean(),
            cv_rmse.mean(),
            cv_mae.mean(),
        ],
        "std": [
            cv_r2.std(ddof=1),
            cv_rmse.std(ddof=1),
            cv_mae.std(ddof=1),
        ],
    }
)

# Özet tabloyu gösteriyoruz.
display(cv_summary)

print("✅ CHECKPOINT 04.10: 5-fold cross-validation tamamlandı.")

In [ ]:
# Eğitim çıktı tablosunu metadata, feature, gerçek ve tahmin değerleriyle oluşturuyoruz.
train_output = pd.concat(
    [
        train_meta.reset_index(drop=True),
        X_train.reset_index(drop=True),
    ],
    axis=1,
)

# Eğitim gerçek ve tahmin sütunlarını ekliyoruz.
train_output["actual_pIC50"] = y_train.to_numpy()
train_output["predicted_pIC50"] = train_predictions
train_output["residual"] = train_output["actual_pIC50"] - train_output["predicted_pIC50"]
train_output["split"] = "train"

# Test çıktı tablosunu metadata, feature, gerçek ve tahmin değerleriyle oluşturuyoruz.
test_output = pd.concat(
    [
        test_meta.reset_index(drop=True),
        X_test.reset_index(drop=True),
    ],
    axis=1,
)

# Test gerçek ve tahmin sütunlarını ekliyoruz.
test_output["actual_pIC50"] = y_test.to_numpy()
test_output["predicted_pIC50"] = test_predictions
test_output["residual"] = test_output["actual_pIC50"] - test_output["predicted_pIC50"]
test_output["split"] = "test"

# Çıktı yollarını tanımlıyoruz.
MODEL_OUTPUT = MODEL_DIR / "CHEMBL206_final_tree_model.joblib"
FEATURE_NAMES_OUTPUT = MODEL_DIR / "CHEMBL206_final_feature_names.json"
METADATA_OUTPUT = MODEL_DIR / "CHEMBL206_final_model_metadata.json"
TRAIN_OUTPUT = MODEL_DIR / "CHEMBL206_train_with_predictions.csv"
TEST_OUTPUT = MODEL_DIR / "CHEMBL206_test_with_predictions.csv"
METRICS_OUTPUT = MODEL_DIR / "CHEMBL206_final_model_metrics.csv"
CV_OUTPUT = MODEL_DIR / "CHEMBL206_final_model_5CV.csv"

# Final modeli kaydediyoruz.
joblib.dump(final_model, MODEL_OUTPUT)

# Feature adlarını JSON olarak kaydediyoruz.
with open(FEATURE_NAMES_OUTPUT, "w", encoding="utf-8") as file:
    json.dump(feature_columns, file, ensure_ascii=False, indent=2)

# Model metadata sözlüğünü hazırlıyoruz.
model_metadata = {
    "selected_model_name": selected_model_name,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(len(feature_columns)),
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
}

# Model metadata bilgisini kaydediyoruz.
with open(METADATA_OUTPUT, "w", encoding="utf-8") as file:
    json.dump(model_metadata, file, ensure_ascii=False, indent=2)

# Tahmin tablolarını kaydediyoruz.
train_output.to_csv(TRAIN_OUTPUT, index=False)
test_output.to_csv(TEST_OUTPUT, index=False)

# Performans tablolarını kaydediyoruz.
metrics_table.to_csv(METRICS_OUTPUT, index=True)
cv_summary.to_csv(CV_OUTPUT, index=False)

print("Final model:", MODEL_OUTPUT)
print("Feature adları:", FEATURE_NAMES_OUTPUT)
print("Model metadata:", METADATA_OUTPUT)
print("Train tahminleri:", TRAIN_OUTPUT)
print("Test tahminleri:", TEST_OUTPUT)
print("Metrikler:", METRICS_OUTPUT)
print("5CV özeti:", CV_OUTPUT)

# Kritik dosyaların oluştuğunu kontrol ediyoruz.
assert MODEL_OUTPUT.exists()
assert TRAIN_OUTPUT.exists()
assert TEST_OUTPUT.exists()

print("✅ CHECKPOINT 04.11: Model, bölünmeler, tahminler ve metrikler kaydedildi.")

In [ ]:
# Gerçek ve tahmin pIC50 değerlerini görsel olarak karşılaştırıyoruz.
plt.figure(figsize=(7, 7))

# Eğitim noktalarını çiziyoruz.
plt.scatter(
    train_output["actual_pIC50"],
    train_output["predicted_pIC50"],
    alpha=0.55,
    label="Train",
)

# Test noktalarını çiziyoruz.
plt.scatter(
    test_output["actual_pIC50"],
    test_output["predicted_pIC50"],
    alpha=0.75,
    label="Test",
)

# İdeal y=x doğrusunun sınırlarını belirliyoruz.
axis_min = min(
    train_output["actual_pIC50"].min(),
    test_output["actual_pIC50"].min(),
    train_output["predicted_pIC50"].min(),
    test_output["predicted_pIC50"].min(),
)

axis_max = max(
    train_output["actual_pIC50"].max(),
    test_output["actual_pIC50"].max(),
    train_output["predicted_pIC50"].max(),
    test_output["predicted_pIC50"].max(),
)

# İdeal tahmin doğrusunu çiziyoruz.
plt.plot([axis_min, axis_max], [axis_min, axis_max], linestyle="--")

# Eksen ve başlıkları ekliyoruz.
plt.xlabel("Gerçek pIC50")
plt.ylabel("Tahmin edilen pIC50")
plt.title(f"CHEMBL206 — {selected_model_name}")
plt.legend()
plt.tight_layout()

# Grafiği kaydediyoruz.
PREDICTION_PLOT = REPORT_DIR / "CHEMBL206_actual_vs_predicted.png"
plt.savefig(PREDICTION_PLOT, dpi=300, bbox_inches="tight")
plt.show()

print("Tahmin grafiği:", PREDICTION_PLOT)
print("✅ CHECKPOINT 04.12: Gerçek–tahmin grafiği oluşturuldu.")

## Ders sonu değerlendirme soruları

1. LazyPredict sonuç tablosundaki en iyi model neden doğrudan “nihai model” sayılmamalıdır?
2. Train R² çok yüksek, test R² belirgin düşükse hangi problemden şüpheleniriz?
3. RMSE ile MAE arasındaki fark nedir?
4. Test seti hiperparametre seçimi için kullanılırsa neden iyimser performans oluşur?

**Sonraki notebook:** Leverage tabanlı application domain, Williams plot ve SHAP açıklanabilirlik analizi.